In [1]:
import sys
from pathlib import Path

# Move up one level from 'notebooks' to the root directory and add it to sys.path
root_dir = Path.cwd().parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

In [2]:
from functools import partial

import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader

from src.data.split import split_train_validation
from src.preprocessing.text import build_vocabulary
from src.deep_learning.dataset import (
    ToxicCommentDataset,
    collate_fn,
)
from src.deep_learning.model import ToxicCommentBiLSTM

In [3]:
# 1. Split the dataset.
train_df, val_df = split_train_validation()

# 2. Build vocabulary using training comments only.
vocab = build_vocabulary(
    train_df["comment_text"],
    min_freq=2,
    max_vocab_size=30_000,
)

# 3. Create training dataset.
train_dataset = ToxicCommentDataset(
    train_df,
    vocab,
    max_length=300,
)

# 4. Create validation dataset.
val_dataset = ToxicCommentDataset(
    val_df,
    vocab,
    max_length=300,
)

# 5. Create training DataLoader.
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    collate_fn=partial(collate_fn, pad_id=vocab.pad_id),
)

# 6. Create validation DataLoader.
val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=partial(collate_fn, pad_id=vocab.pad_id),
)

In [5]:
print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Vocabulary size:", len(vocab))

Training samples: 127656
Validation samples: 31915
Training batches: 3990
Validation batches: 998
Vocabulary size: 30000


In [4]:
# Get one training batch.
train_batch = next(iter(train_loader))

# Get one validation batch.
val_batch = next(iter(val_loader))

print("TRAINING BATCH")
print("Input IDs:", train_batch["input_ids"].shape)
print("Labels:", train_batch["labels"].shape)

print("\nVALIDATION BATCH")
print("Input IDs:", val_batch["input_ids"].shape)
print("Labels:", val_batch["labels"].shape)

TRAINING BATCH
Input IDs: torch.Size([32, 300])
Labels: torch.Size([32, 6])

VALIDATION BATCH
Input IDs: torch.Size([32, 300])
Labels: torch.Size([32, 6])


In [10]:
# Select device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print(f"Using device: {device}")

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: NVIDIA GeForce RTX 2050


In [7]:
# Initialize Model, Loss, and Optimizer-

model = ToxicCommentBiLSTM(
    vocab_size=len(vocab),
    embedding_dim=100,
    hidden_size=128,
    num_labels=6,
    dropout=0.3,
    pad_id=vocab.pad_id,
).to(device)

# Loss and optimizer.
criterion = nn.BCEWithLogitsLoss()

optimizer = Adam(
    model.parameters(),
    lr=0.001,
)

In [8]:
from src.deep_learning.trainer import train_one_epoch, evaluate

# Train for one epoch.
train_loss = train_one_epoch(
    model,
    train_loader,
    criterion,
    optimizer,
    device,
)

print("Training loss:", train_loss)

# Evaluate on validation data.
val_loss = evaluate(
    model,
    val_loader,
    criterion,
    device,
)

print("Validation loss:", val_loss)

Training loss: 0.0793015238360115
Validation loss: 0.056303948632521854


In [11]:
# Experiment	Device	Epochs	Training loss	Validation loss	Time
# BiLSTM v1	    CPU	      1	      0.076618	     0.056350	12 min
# BiLSTM v1	    RTX 2050  1	      0.079302	     0.056304	58 sec